<a href="https://colab.research.google.com/github/joexner/roxene/blob/master/notebooks/training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Authenticate to GCP on a custom runtime
!gcloud auth login --no-launch-browser

# Set your GCP Project ID
PROJECT_ID = 'roxene-0'
!gcloud config set project {PROJECT_ID}



You are running on a Google Compute Engine virtual machine.
It is recommended that you use service accounts for authentication.

You can run:

  $ gcloud config set account `ACCOUNT`

to switch accounts if necessary.

Your credentials may be visible to others with access to this
virtual machine. Are you sure you want to authenticate with
your personal account?

Do you want to continue (Y/n)?  Y

Go to the following link in your browser, and complete the sign-in prompts:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=32555940559.apps.googleusercontent.com&redirect_uri=https%3A%2F%2Fsdk.cloud.google.com%2Fauthcode.html&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fappengine.admin+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcompute+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Faccounts.

In [2]:
%cd
# Check if roxene directory exists, if not, clone it.
!if [ ! -d roxene ]; then git clone https://github.com/joexner/roxene.git; fi

/root
Cloning into 'roxene'...
remote: Enumerating objects: 2730, done.
remote: Counting objects: 100% (392/392), done.
remote: Compressing objects: 100% (168/168), done.
remote: Total 2730 (delta 269), reused 267 (delta 204), pack-reused 2338 (from 1)
Receiving objects: 100% (2730/2730), 1.38 MiB | 8.77 MiB/s, done.
Resolving deltas: 100% (1864/1864), done.


In [3]:
# Change directory to roxene and pull latest changes.
%cd ~/roxene
!git pull
!git log -1 --pretty="%ci: %s"

/root/roxene
Already up to date.
2026-09-01 23:00:15 -0400: Created using Colab


In [ ]:
# The 'alloydb' extra adds the AlloyDB Python Connector and the pg8000 driver
!pip install -e ".[alloydb]"

## AlloyDB

Everything below reaches AlloyDB through the [AlloyDB Python Connector](https://github.com/GoogleCloudPlatform/alloydb-python-connector#sync-psycopg--sqlalchemy),
which resolves the instance from its URI and tunnels over TLS.

Do **not** build a `host:port` URL out of the PSC DNS name and hand it to psycopg2. That name only
resolves inside the connector's tunnel, which is what produced `OperationalError: could not
translate host name`. A single `Connector` is opened here and left open for the whole session,
because the SQLAlchemy pool calls back into it for every new connection.

In [ ]:
# AlloyDB instance, assumed to already exist and be reachable over PSC from this VPC
REGION = 'us-central1'
CLUSTER_NAME = 'roxene-alloy-db-1'
INSTANCE_NAME = 'primary'
INSTANCE_URI = f"projects/{PROJECT_ID}/locations/{REGION}/clusters/{CLUSTER_NAME}/instances/{INSTANCE_NAME}"

DB_USER = 'postgres'
DB_PASSWORD = 'enexor'

# Trial run parameters
POOL_SIZE = 1000
NUM_TRIALS = 1000
NUM_THREADS = 20
NUM_MUTAGENS = 100
BREED_AND_CULL_INTERVAL = 10
SEED = 11235

print(INSTANCE_URI)

In [ ]:
import logging
import time

import sqlalchemy
from google.cloud.alloydbconnector import Connector, IPTypes

logging.basicConfig(level=logging.INFO, format='%(asctime)s - [%(threadName)s]\t- %(name)s: %(message)s', force=True)
logging.getLogger("roxene.tic_tac_toe.environment").setLevel(logging.DEBUG)

# One connector for the whole session. It has to outlive every engine built from it,
# since the SQLAlchemy pool calls back into it for each new connection.
connector = Connector()


def alloydb_engine(db: str, **engine_kwargs) -> sqlalchemy.Engine:
    return sqlalchemy.create_engine(
        "postgresql+pg8000://",
        creator=lambda: connector.connect(
            INSTANCE_URI,
            "pg8000",
            user=DB_USER,
            password=DB_PASSWORD,
            db=db,
            ip_type=IPTypes.PSC,
        ),
        **engine_kwargs,
    )


# A fresh database per run. CREATE DATABASE cannot run inside a transaction, so
# dial the default 'postgres' database with AUTOCOMMIT to issue it.
DB_NAME = f"roxene_{int(time.time())}"
admin_engine = alloydb_engine("postgres", isolation_level="AUTOCOMMIT")
try:
    with admin_engine.connect() as conn:
        exists = conn.execute(
            sqlalchemy.text("SELECT 1 FROM pg_database WHERE datname = :db"),
            {"db": DB_NAME},
        ).scalar()
        if exists:
            print(f"Database {DB_NAME} already exists")
        else:
            conn.execute(sqlalchemy.text(f'CREATE DATABASE "{DB_NAME}"'))
            print(f"Created database {DB_NAME}")
finally:
    admin_engine.dispose()

engine = alloydb_engine(DB_NAME, pool_size=NUM_THREADS)

with engine.connect() as conn:
    print(conn.execute(sqlalchemy.text("SELECT current_database(), now()")).fetchone())

In [ ]:
import importlib
import site
from threading import Thread

# Refresh site packages and invalidate import caches so the kernel sees the new package
site.main()
importlib.invalidate_caches()

from numpy.random import Generator, default_rng

from roxene import EntityBase
from roxene.tic_tac_toe.environment import Environment
from roxene.util import set_rng

logger = logging.getLogger("roxene.training")

EntityBase.metadata.create_all(engine)

logger.info(f"Seed={SEED}")
main_rng: Generator = default_rng(SEED)
set_rng(main_rng)
env = Environment(engine)

logger.info(f"Populating environment with {POOL_SIZE} organisms and {NUM_MUTAGENS} mutagens")
env.populate(POOL_SIZE)
env.add_mutagens(NUM_MUTAGENS)
logger.info("Done populating environment")

# Replace 5% of the herd at a time, up to 5
num_to_cull = num_to_breed = int(max(POOL_SIZE * .05, 5))


def run_trials(worker_trials: int, worker_rng: Generator):
    set_rng(worker_rng)
    for iteration in range(worker_trials):
        logger.info("Building trial")
        trial = env.start_trial()
        logger.info(f"Starting trial, {env.count_trials(True, False)} trials running, {env.count_trials()} trials total")
        trial.run()
        logger.info("Trial complete, saving results")
        env.complete_trial(trial)
        logger.info(f"Finished trial {trial} with {len(trial.moves)} moves")
        if iteration % BREED_AND_CULL_INTERVAL == 0 and iteration > 0:
            logger.info("Culling")
            env.cull(num_to_cull)
            logger.info("Done culling, breeding")
            env.breed(num_to_breed)
            logger.info("Done breeding")


# Estimate, the total could be over by (NUM_THREADS - 1)
trials_per_thread = int((NUM_TRIALS - 1) / NUM_THREADS) + 1
rngs = main_rng.spawn(NUM_THREADS)
threads = []

for i in range(NUM_THREADS):
    logger.info(f"Starting thread {i}")
    thread = Thread(target=run_trials, args=(trials_per_thread, rngs.pop()))
    thread.start()
    threads.append(thread)

for thread in threads:
    thread.join()

logger.info(f"Done, {env.count_trials()} trials total")

In [ ]:
# Close the connector last, after every engine that dials through it is disposed
engine.dispose()
connector.close()